In [23]:
import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [24]:
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

In [25]:
train_data = datasets.ImageFolder(root = 'train', transform = transform)
test_data = datasets.ImageFolder(root = 'test', transform = transform)

train_loader = DataLoader(train_data, batch_size = 32, shuffle = True)
test_loader = DataLoader(test_data, batch_size = 32)

In [26]:
class DogCat(nn.Module):
    def __init__(self):
        super(DogCat, self).__init__()
        
        self.conv_layer = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            
            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            
            nn.Conv2d(64, 128, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        
        self.fc_layer = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 16 * 16, 128),
            nn.ReLU(),
            nn.Linear(128, 1),
            nn.Sigmoid()
        )
    
    def forward(self, x):
        x = self.conv_layer(x)
        x = self.fc_layer(x)
        
        return x

In [27]:
model = DogCat()

In [28]:
model.eval()

DogCat(
  (conv_layer): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU()
    (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (fc_layer): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=32768, out_features=128, bias=True)
    (2): ReLU()
    (3): Linear(in_features=128, out_features=1, bias=True)
    (4): Sigmoid()
  )
)

In [29]:
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr = 0.001)

In [30]:
epochs = 10

In [31]:
for epoch in range(epochs):
    
    model.train()
    running_loss = 0
    
    for images, labels in train_loader:
        
        labels = labels.float().unsqueeze(1)
        output = model(images)
        loss = criterion(output, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    
    print(f'Epoch {epoch+1}/{epochs}, Loss: {running_loss/len(train_loader)}')

Epoch 1/10, Loss: 0.6983497578000265
Epoch 2/10, Loss: 0.6782206571291364
Epoch 3/10, Loss: 0.6284354177732316
Epoch 4/10, Loss: 0.5792177447250911
Epoch 5/10, Loss: 0.5146268653491187
Epoch 6/10, Loss: 0.4700847575588832
Epoch 7/10, Loss: 0.3747178122164711
Epoch 8/10, Loss: 0.30219016235972207
Epoch 9/10, Loss: 0.21313938599020715
Epoch 10/10, Loss: 0.1386753949262793


In [32]:
with torch.no_grad():
    model.eval()
    correct = 0
    total = 0
    
    for images, labels in test_loader:
        output = model(images)
        predicted = (output > 0.5).float()
        total += labels.size(0)
        correct += (predicted.squeeze() == labels.float()).sum().item()
    
    print(f'Accuracy: {correct/total * 100:.2f}%')

Accuracy: 71.38%


In [33]:
with torch.no_grad():
    model.eval()
    correct = 0
    total = 0
    
    for images, labels in train_loader:
        output = model(images)
        predicted = (output > 0.5).float()
        total += labels.size(0)
        correct += (predicted.squeeze() == labels.float()).sum().item()
    
    print(f'Accuracy: {correct/total * 100:.2f}%')

Accuracy: 96.60%


In [34]:
model.eval()

DogCat(
  (conv_layer): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU()
    (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (fc_layer): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=32768, out_features=128, bias=True)
    (2): ReLU()
    (3): Linear(in_features=128, out_features=1, bias=True)
    (4): Sigmoid()
  )
)

NameError: name 'classes' is not defined